# Linear Regression — King County House Prices

Run this notebook with `KC_House_Data.csv` in the same folder. It calculates MAE, MSE, RMSE and especially R² on test data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("KC_House_Data.csv")
print("Dataset Shape:", df.shape)
display(df.head())
print("\nMissing values:")
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df=df.drop_duplicates()
df["date"]=pd.to_datetime(df["date"], errors="coerce")
df["sale_year"]=df["date"].dt.year
df["sale_month"]=df["date"].dt.month
df["sale_day"]=df["date"].dt.day
df=df.drop(columns=["date"])
if "id" in df.columns: df=df.drop(columns=["id"])

In [ ]:
y=df["price"]
X=df.drop(columns=["price"])
if "zipcode" in X.columns:
    X["zipcode"]=X["zipcode"].astype(str)
    X=pd.get_dummies(X, columns=["zipcode"], drop_first=True)
print("X:", X.shape, "y:", y.shape)

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
print("Train:",X_train.shape,"Test:",X_test.shape)

In [ ]:
model=LinearRegression()
model.fit(X_train_scaled,y_train)
y_pred=model.predict(X_test_scaled)
mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)
rmse=np.sqrt(mse)
r2=r2_score(y_test,y_pred)
print(f"MAE: {mae:,.2f}")
print(f"MSE: {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R² Score: {r2:.4f}")
print(f"R² Percentage: {r2*100:.2f}%")

In [ ]:
results=pd.DataFrame({"Actual_Price":y_test.values,"Predicted_Price":y_pred})
results["Prediction_Error"]=results["Actual_Price"]-results["Predicted_Price"]
display(results.head(20))

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(y_test,y_pred,alpha=0.5)
lo=min(y_test.min(),y_pred.min()); hi=max(y_test.max(),y_pred.max())
plt.plot([lo,hi],[lo,hi],linestyle="--",linewidth=2)
plt.xlabel("Actual House Price"); plt.ylabel("Predicted House Price"); plt.title("Linear Regression - Actual vs Predicted"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
residuals=y_test.values-y_pred
plt.figure(figsize=(8,6))
plt.scatter(y_pred,residuals,alpha=0.5)
plt.axhline(0,linestyle="--",linewidth=2)
plt.xlabel("Predicted House Price"); plt.ylabel("Prediction Error"); plt.title("Linear Regression - Prediction Error"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
results.to_csv("Linear_Regression_house_price_predictions.csv",index=False)
print("Saved: Linear_Regression_house_price_predictions.csv")